# Definining Cohort

**The goal of this notebook is to identify patients with advanced non-small cell lung cancer who received first-line treatment with pembrolizumab plus platinum-based chemotherapy or pembrolizumab. Those who received targeted therapy in later lines are removed.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## 1. Pembrolizumab plus platinum-based chemotherapy

In [3]:
therapy = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
(
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .LineName.value_counts()
    .head(20)
)

LineName
Carboplatin,Paclitaxel                                12602
Carboplatin,Pembrolizumab,Pemetrexed                  10174
Carboplatin,Pemetrexed                                 7573
Pembrolizumab                                          7207
Osimertinib                                            3952
Erlotinib                                              3305
Bevacizumab,Carboplatin,Pemetrexed                     3232
Clinical Study Drug                                    3008
Carboplatin,Paclitaxel,Pembrolizumab                   2466
Nivolumab                                              2303
Carboplatin,Paclitaxel Protein-Bound                   2192
Bevacizumab,Carboplatin,Paclitaxel                     1465
Carboplatin,Gemcitabine                                1385
Carboplatin,Paclitaxel Protein-Bound,Pembrolizumab     1304
Pemetrexed                                             1240
Cisplatin,Pemetrexed                                    963
Cisplatin,Etoposide            

In [5]:
therapy_1l = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
)

In [6]:
targeted_agents = [
    # EGFR
    'Osimertinib', 
    'Erlotinib', 
    'Gefitinib', 
    'Afatinib',
    'Amivantamab', 

    # ALK
    'Dacomitinib',
    'Crizotinib', 
    'Ceritinib', 
    'Alectinib', 
    'Brigatinib', 
    'Lorlatinib',

    # ROS1
    'Repotrectinib', 
    'Taletrectinib', 
    'Entrectinib', 
    'Crizotinib',

    # RET
    'Selpercatinib', 
    'Pralsetinib', 

    # BRAF
    'Dabrafenib',
    'Trametinib', 
    'Encorafenib',
    'Binimetinib',

    # NTRK
    'Larotrectinib',
    'Repotrectinib',

    # NRG1
    'Zenocutuzumab',

    # RAS
    'Sotorasib',
    'Adagrasib'
]

pattern = "|".join(targeted_agents)

In [7]:
(
    therapy_1l[
    therapy_1l['LineName'].str.contains('Pembrolizumab')
    & therapy_1l['LineName'].str.contains('Carboplatin|Cisplatin')
    & ~therapy_1l['LineName'].str.contains('Clinical Study Drug')
    & ~therapy_1l['LineName'].str.contains(pattern)]
    .LineName.value_counts().head(10)
)

LineName
Carboplatin,Pembrolizumab,Pemetrexed                  10174
Carboplatin,Paclitaxel,Pembrolizumab                   2466
Carboplatin,Paclitaxel Protein-Bound,Pembrolizumab     1304
Cisplatin,Pembrolizumab,Pemetrexed                      101
Carboplatin,Paclitaxel,Pembrolizumab,Pemetrexed          55
Carboplatin,Pembrolizumab                                49
Carboplatin,Gemcitabine,Pembrolizumab                    40
Carboplatin,Docetaxel,Pembrolizumab                      29
Cisplatin,Gemcitabine,Pembrolizumab                      17
Carboplatin,Cemiplimab,Pembrolizumab,Pemetrexed          12
Name: count, dtype: int64

In [8]:
pembro_plat_df = (
    therapy_1l[
    therapy_1l['LineName'].str.contains('Pembrolizumab')
    & therapy_1l['LineName'].str.contains('Carboplatin|Cisplatin')
    & ~therapy_1l['LineName'].str.contains('Clinical Study Drug')
    & ~therapy_1l['LineName'].str.contains(pattern)]
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro_platinum'))

In [9]:
pembro_plat_df.sample(3)

,PatientID,LineName,StartDate
39193,F0E61ECC8D33A,pembro_platinum,2020-03-09
5350,F368FB74C3B88,pembro_platinum,2022-12-12
127619,FBB80ADC184A5,pembro_platinum,2021-06-04


In [10]:
row_ID(pembro_plat_df)

(14463, 14463)

## 2. Pembrolizumab 

In [11]:
pembro_df = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Pembrolizumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro'))

In [12]:
pembro_df.sample(3)

,PatientID,LineName,StartDate
111483,F8C373BE87465,pembro,2019-03-12
53884,FA69DED72A4D0,pembro,2017-01-05
129745,F018EDCD6F208,pembro,2018-10-01


In [13]:
row_ID(pembro_df)

(7207, 7207)

## 3. IDs for patients with targeted agents

In [14]:
targeted_ids = (
    therapy
    .query('LineName.str.contains(@pattern, case=False, na=False)', engine="python")
    .PatientID
)

## 4. Combine dataframes and export to csv 

In [15]:
firstline_pembrochemo_pembro_index = pd.concat([pembro_plat_df, pembro_df], axis = 0)

In [16]:
row_ID(firstline_pembrochemo_pembro_index)

(21670, 21670)

In [17]:
# Remove patients any future targeted agent recepit 
firstline_pembrochemo_pembro_index = firstline_pembrochemo_pembro_index[~firstline_pembrochemo_pembro_index.PatientID.isin(targeted_ids)]

In [18]:
row_ID(firstline_pembrochemo_pembro_index)

(20623, 20623)

In [19]:
firstline_pembrochemo_pembro_index.sample(3)

,PatientID,LineName,StartDate
98342,FCC64803DC695,pembro,2020-10-21
92648,FC69886361D26,pembro_platinum,2021-09-21
122553,F9BFDD26B4CF9,pembro_platinum,2023-02-27


In [20]:
firstline_pembrochemo_pembro_index.LineName.value_counts()

LineName
pembro_platinum    13619
pembro              7004
Name: count, dtype: int64

In [21]:
firstline_pembrochemo_pembro_index.to_csv('../outputs/pembrochemo_pembro_index.csv', index = False)